In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive', force_remount=True)
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto MLDM')

    GIT_BRANCH = 'incremento-punti-dataset'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

# 02 - Download e Riepilogo dei dati satellitari per l'area di studio
Download delle 6 bande Sentinel-2 per l'intera Bounding Box della Capitanata per ciascun mese degli anni selezionati.

#### Download dei dati satellitari (Bounding Box Capitanata)

In [ ]:
import rasterio
from src.config import CAPITANATA_BBOX, YEARS_TO_FETCH
from src.test_api2 import download_area

out_directory = DATA_DIR / "processed" / "sentinel2_capitanata_area"

print(f"BBox esatta Capitanata: {CAPITANATA_BBOX}")
print(f"Anni target selezionati: {YEARS_TO_FETCH}")

# avvio del download
downloaded_files = download_area(
    name="capitanata",
    bbox=CAPITANATA_BBOX,
    years=YEARS_TO_FETCH,
    months=range(1, 13),
    out_dir=str(out_directory)
)

# riepilogo dei dati scaricati
tif_files = sorted(list(out_directory.glob("*.tif")))

print("\n" + "-" * 50)
print("Riepilogo dati satellitari Capitanata")
print("-" * 50)
print(f"Cartella: {out_directory.resolve()}")
print(f"File GeoTIFF mensili trovati: {len(tif_files)} (attesi: {len(YEARS_TO_FETCH) * 12})")

total_size_mb = 0.0
for f in tif_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    total_size_mb += size_mb
    print(f"  • {f.name} ({size_mb:.1f} MB)")

print(f"Spazio totale occupato: {total_size_mb / 1024:.2f} GB")

# controllo di integrità sul primo file disponibile
if tif_files:                                                
    with rasterio.open(tif_files[0]) as src:                 
        print(f"\nVerifica formato raster ({tif_files[0].name}):")                                                      
        print(f"  • Dimensioni: {src.width} x {src.height} pixel")                                                        
        print(f"  • Numero bande: {src.count}") 